# LangChain Multi-Session Example

This notebook demonstrates how to maintain multiple sessions and preserve context for each session in LangChain. It includes a project aim section based on the provided image and code to manage session-specific conversation history.

## Project Aim

The project goal is to build a multi-session conversational assistant that can keep separate context for each user session. The image provided with the request should be treated as the project aim description for building session-aware dialogue.

## 1. Install and Import Dependencies

Install the required LangChain packages and import the modules needed for multi-session context management.

In [7]:
# Uncomment the install line if you are running this notebook in a fresh environment
# !pip install langchain-core langchain-openrouter langchain-community

import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import Runnable, RunnableLambda, ConfigurableFieldSpec

# Define the OpenRouter model wrapper
model = ChatOpenRouter(
    model="openai/gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=400,
    api_key=os.getenv("OPENROUTER_API_KEY")
)


## 2. Multi-Session Context Design

We maintain session-specific chat history using a dictionary keyed by `session_id`. Each session stores its own `InMemoryChatMessageHistory`, so context is isolated between users or browser sessions.

In [8]:
# Session store for multiple independent conversations
session_histories: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in session_histories:
        session_histories[session_id] = InMemoryChatMessageHistory()
    return session_histories[session_id]


def clear_session(session_id: str) -> None:
    if session_id in session_histories:
        session_histories[session_id].clear()


# Example of dynamic session creation
# Sessions are created when first referenced, so this works for any number of sessions.
session_ids = ["session_alice", "session_bob", "session_charlie"]
for sid in session_ids:
    get_session_history(sid).add_user_message(f"Hello from {sid}!")

print("Sessions created:", list(session_histories.keys()))


Sessions created: ['session_alice', 'session_bob', 'session_charlie']


## 3. Session-Aware Prompt and Pipeline

Each session uses its own history. The pipeline will insert the correct chat history for the current session, preserving context without leaking between sessions.

In [9]:
system_prompt = SystemMessagePromptTemplate.from_template(
    "You are a helpful assistant that remembers the user's session history."
)

human_prompt = HumanMessagePromptTemplate.from_template(
    "The user asks: {user_input}"
)

chat_prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    MessagesPlaceholder(variable_name="history"),
    human_prompt,
])

session_pipeline = chat_prompt | model


def invoke_session(session_id: str, user_input: str):
    history = get_session_history(session_id)
    history.add_user_message(user_input)
    response = session_pipeline.invoke({"user_input": user_input, "history": history.messages})
    history.add_ai_message(response.content)
    return response


# Example usage for multiple sessions
for sid, prompt_text in [
    ("session_alice", "What is LangChain?"),
    ("session_bob", "Explain session history in simple terms."),
    ("session_charlie", "Summarize how session data is stored."),
]:
    response = invoke_session(sid, prompt_text)
    print(f"{sid} response:", response.content)


session_alice response: LangChain is a decentralized language learning platform that uses blockchain technology to connect language learners with tutors around the world. It provides a secure and transparent way for users to find language tutors, schedule lessons, and make payments using cryptocurrency. LangChain aims to create a global community of language learners and tutors, enabling people to learn new languages in a flexible and efficient manner.
session_bob response: Session history refers to the record of actions and interactions that a user has performed during their current session on a website or application. This can include pages visited, searches made, items clicked on, and other activities that the user has engaged in while using the platform. The session history helps track the user's journey and can be used to personalize their experience or provide relevant recommendations based on their past interactions.
session_charlie response: Session data is typically stored on 

## 4. Inspect and Reset Sessions

Show how to inspect each session's stored context and how to clear a session when it ends.

In [10]:
print("Session Alice history:")
for msg in session_histories["session_alice"].messages:
    print(msg.type, msg.content)

print("\nSession Bob history:")
for msg in session_histories["session_bob"].messages:
    print(msg.type, msg.content)

# Clear Bob's session to end it
clear_session("session_bob")
print("\nSessions after clearing Bob:", list(session_histories.keys()))


Session Alice history:
human Hello from session_alice!
human What is LangChain?
ai LangChain is a decentralized language learning platform that uses blockchain technology to connect language learners with tutors around the world. It provides a secure and transparent way for users to find language tutors, schedule lessons, and make payments using cryptocurrency. LangChain aims to create a global community of language learners and tutors, enabling people to learn new languages in a flexible and efficient manner.

Session Bob history:
human Hello from session_bob!
human Explain session history in simple terms.
ai Session history refers to the record of actions and interactions that a user has performed during their current session on a website or application. This can include pages visited, searches made, items clicked on, and other activities that the user has engaged in while using the platform. The session history helps track the user's journey and can be used to personalize their expe